# WavLM Utterance-Level Extraction (GPU) - SAFE VERSION v2
## Fixed: CUDA memory + device placement + scoping bug
**Changes:**
1. `batch_size=1` (one video at a time)
2. Explicit `.to(device)` for ALL tensors
3. Chunked audio processing for long videos
4. `torch.cuda.empty_cache()` after each video
5. Checkpoint every 5 videos

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive')
print('Drive mounted')

In [ ]:
# Install dependencies
!pip install -q transformers datasets torch torchaudio librosa

In [ ]:
import os
import json
import torch
import numpy as np
from transformers import WavLMModel, WavLMConfig
import librosa
from tqdm import tqdm

# Setup
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SR = 16000
OUTPUT_DIR = '/content/gdrive/MyDrive/wavlm_utterance_safe'
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, 'checkpoint.json')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Device: {DEVICE}')

In [ ]:
# Load WavLM model ONCE
print('Loading WavLM model...')
wavlm_model = WavLMModel.from_pretrained('microsoft/wavlm-base-plus').to(DEVICE)
wavlm_model.eval()
print('Model loaded on', DEVICE)

In [ ]:
# Download utterance labels
print('Loading utterances...')
url = 'https://raw.githubusercontent.com/Das-rebel/chuck-audio-notebooks/main/utterances_clean.jsonl.gz'
!wget -q -O /tmp/utterances.jsonl.gz {url}
!gunzip -f /tmp/utterances.jsonl.gz

utterances = []
with open('/tmp/utterances.jsonl') as f:
    for line in f:
        utterances.append(json.loads(line))
print(f'Loaded {len(utterances)} utterances')

# Group by video
video_utts = {}
for u in utterances:
    vid = u['video_id']
    if vid not in video_utts:
        video_utts[vid] = []
    video_utts[vid].append(u)
print(f'Videos: {len(video_utts)}')

In [ ]:
# Find audio files across ALL subfolders
audio_base = '/content/gdrive/MyDrive/chuckle_audio_all'
audio_files = {}

for root, dirs, files in os.walk(audio_base):
    for f in files:
        if f.endswith('.mp3') or f.endswith('.wav'):
            vid = os.path.splitext(f)[0]
            audio_files[vid] = os.path.join(root, f)

# Also check MyDrive/audio and MyDrive/chuckle_audio
for base in ['/content/gdrive/MyDrive/audio', '/content/gdrive/MyDrive/chuckle_audio']:
    if os.path.exists(base):
        for root, dirs, files in os.walk(base):
            for f in files:
                if f.endswith('.mp3') or f.endswith('.wav'):
                    vid = os.path.splitext(f)[0]
                    if vid not in audio_files:
                        audio_files[vid] = os.path.join(root, f)

print(f'Audio files found: {len(audio_files)}')

# Find videos with both audio AND utterances
videos_to_process = [v for v in video_utts.keys() if v in audio_files]
print(f'Videos with audio + labels: {len(videos_to_process)}')

In [ ]:
# Load checkpoint
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        checkpoint = json.load(f)
    extracted_ids = set(checkpoint.get('extracted_ids', []))
    print(f'Resuming from {len(extracted_ids)} already extracted')
else:
    extracted_ids = set()
    checkpoint = {'extracted_ids': [], 'errors': []}

videos_to_process = [v for v in videos_to_process if v not in extracted_ids]
print(f'Starting fresh: {len(videos_to_process)} videos to process')

In [ ]:
def extract_embeddings(audio_tensor, start_sample, end_sample, model, max_chunk_seconds=60):
    """Extract WavLM embedding for a single utterance segment."""
    segment = audio_tensor[start_sample:end_sample]
    
    duration = len(segment) / SR
    
    if duration < 0.1:  # Skip very short
        return None
    
    # Handle long segments by chunking
    if duration > max_chunk_seconds:
        chunk_embeddings = []
        for chunk_start in np.arange(0, duration, max_chunk_seconds):
            chunk_end = min(chunk_start + max_chunk_seconds, duration)
            c_start = int(chunk_start * SR)
            c_end = int(chunk_end * SR)
            chunk = audio_tensor[c_start:c_end]
            if len(chunk) < SR * 0.1:
                continue
            chunk = chunk.to(DEVICE)
            with torch.no_grad():
                outputs = model(chunk.unsqueeze(0))
                emb = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
                chunk_embeddings.append(emb)
        
        if not chunk_embeddings:
            return None
        return np.mean(chunk_embeddings, axis=0)
    
    # Normal case
    segment = segment.to(DEVICE)
    with torch.no_grad():
        outputs = model(segment.unsqueeze(0))
        emb = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    
    return emb

In [ ]:
# Process videos ONE AT A TIME
errors = checkpoint.get('errors', [])
extracted = list(extracted_ids)

print(f'Processing {len(videos_to_process)} videos...')
print(f'Estimated time: {len(videos_to_process) * 30 / 60:.1f} minutes')

pbar = tqdm(videos_to_process, desc='Videos')
for i, video_id in enumerate(pbar):
    try:
        utts = video_utts[video_id]
        audio_path = audio_files[video_id]
        
        # Load audio
        audio, sr = librosa.load(audio_path, sr=SR)
        audio_tensor = torch.tensor(audio, dtype=torch.float32)
        del audio  # Free memory
        
        embeddings = []
        
        for utt in utts:
            start = float(utt['start'])
            end = float(utt['end'])
            start_sample = int(start * SR)
            end_sample = int(end * SR)
            
            emb = extract_embeddings(audio_tensor, start_sample, end_sample, wavlm_model)
            
            if emb is not None:
                embeddings.append({
                    'start': start,
                    'end': end,
                    'embedding': emb.tolist()
                })
        
        # Save result
        result = {'video_id': video_id, 'embeddings': embeddings}
        out_path = os.path.join(OUTPUT_DIR, f'{video_id}.json')
        with open(out_path, 'w') as f:
            json.dump(result, f)
        
        extracted.append(video_id)
        
        # Cleanup
        del audio_tensor
        torch.cuda.empty_cache()
        
    except Exception as e:
        errors.append({'video_id': video_id, 'error': str(e)})
        print(f"Error {video_id}: {e}")
        torch.cuda.empty_cache()
    
    # Update checkpoint every 5 videos
    if (i + 1) % 5 == 0:
        checkpoint = {'extracted_ids': extracted, 'errors': errors}
        with open(CHECKPOINT_PATH, 'w') as f:
            json.dump(checkpoint, f)
        pbar.set_postfix({'extracted': len(extracted), 'errors': len(errors)})

# Final checkpoint
checkpoint = {'extracted_ids': extracted, 'errors': errors}
with open(CHECKPOINT_PATH, 'w') as f:
    json.dump(checkpoint, f)

print(f'\nDone! Extracted: {len(extracted)}, Errors: {len(errors)}')

In [ ]:
# Verify extraction
import os

output_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.json') and f != 'checkpoint.json']
print(f'Extracted videos: {len(output_files)}')

# Check sample
if output_files:
    sample = os.path.join(OUTPUT_DIR, output_files[0])
    with open(sample) as f:
        data = json.load(f)
    print(f'Sample: {data["video_id"]}, embeddings: {len(data["embeddings"])}')